In [ ]:
import os
import boto3
import rasterio

import geopandas as gpd
import pandas as pd

from tqdm.notebook import tqdm
from shapely.geometry import box

In [ ]:
in_file_list = "/home/wb411133/temp/fathom_files.csv"

s3_bucket = "wbg-geography01"
s3_prefix = "FATHOM/v2023/"
s3_folder = (
    "GLOBAL-1ARCSEC-NW_OFFSET-1in1000-COASTAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.0"
)

in_files = pd.read_csv(in_file_list)
s3 = boto3.resource("s3")
my_bucket = s3.Bucket(s3_bucket)

In [ ]:
# Read and process from disk
all_extents = []
in_folder = "/home/wb411133/temp/FATHOM/FATHOM_TESTING"
for tile_file in tqdm(os.listdir(in_folder)):
    inR = rasterio.open(os.path.join(in_folder, tile_file))
    shp = box(*inR.bounds)
    all_extents.append([tile_file.replace(".tif", ""), shp])

In [ ]:
fluvial_tiles = gpd.GeoDataFrame(
    pd.DataFrame(all_extents, columns=["ID", "geometry"]), geometry="geometry", crs=4326
)

In [ ]:
# Add a column to extents file indicating if a coastal cell is present
cur_out_folder = os.path.join(
    s3_prefix,
    "GLOBAL-1ARCSEC-NW_OFFSET-1in1000-COASTAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.0",
)
# Get a list of all tiles in a coastal folder
all_files = []
for obj in tqdm(my_bucket.objects.filter(Prefix=cur_out_folder)):
    all_files.append(os.path.basename(obj.key).replace(".tif", ""))

# Add a boolean flag in the fluvial catalog if there is a corresponding coastal tile
fluvial_tiles["COASTAL"] = 0
for idx, row in tqdm(xx.iterrows()):  # noqa
    if row["ID"] in all_files:
        fluvial_tiles.loc[idx, "COASTAL"] = 1

# There are ~300 coastal tiles that don't have a corresponding fluvial tile
coastal_missing = [x for x in all_files if x not in xx["ID"].values]  # noqa
len(coastal_missing)

In [ ]:
# USE AWS CLI to download the missing tiles
for c_tile in coastal_missing:
    aws_command = f"aws s3 cp s3://wbg-geography01/FATHOM/v2023/GLOBAL-1ARCSEC-NW_OFFSET-1in1000-COASTAL-DEFENDED-DEPTH-2020-PERCENTILE50-v3.0/{c_tile}.tif ."
    print(aws_command)
    print(" ")

In [ ]:
# Loop through downloaded files to get extents
missing_tiles_folder = "/home/wb411133/temp/FATHOM/COASTAL_MISSING"
all_extents = []

for tile_file in tqdm(os.listdir(missing_tiles_folder)):
    inR = rasterio.open(os.path.join(missing_tiles_folder, tile_file))
    shp = box(*inR.bounds)
    all_extents.append([tile_file.replace(".tif", ""), shp])
coastal_missing_extents = gpd.GeoDataFrame(
    pd.DataFrame(all_extents, columns=["ID", "geometry"]), geometry="geometry", crs=4326
)

In [ ]:
# Add corresponding flags to final version
fluvial_tiles["FLUVIAL"] = 1
coastal_missing_extents["FLUVIAL"] = 0
coastal_missing_extents["COASTAL"] = 1
combo_extents = fluvial_tiles.append(coastal_missing_extents)

In [ ]:
combo_extents.head()

In [ ]:
combo_extents.tail()

In [ ]:
combo_extents.to_file(
    "/home/wb411133/temp/fathom_tile_extents.geojson", driver="GeoJSON"
)